<!-- AUTOGENERATED-DOCS -->
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Simulation-Benchmarks/rotating-cylinders/main?labpath=notebooks%2Frotating_cylinders.ipynb)

# Rotating Cylinders Benchmark (Taylor-Couette Flow)

## Problem description

This benchmark simulates the flow of a viscous incompressible fluid confined
between two concentric cylinders. The inner cylinder rotates with angular
velocity $\omega_1$ and the outer cylinder with angular velocity $\omega_2$.
For certain parameter combinations the flow is dominated by viscous shear
(Couette flow); above a critical inner-cylinder Reynolds number the flow
becomes unstable and develops the well-known Taylor vortex pattern.

We work in 2D cylindrical coordinates $(r, \theta)$ in the meridional
$(r, z)$ plane. The annulus has inner radius $r_\mathrm{in}$ and outer radius
$r_\mathrm{out}$, with $\eta = r_\mathrm{in}/r_\mathrm{out}$ the radius ratio.

## Governing equations

The incompressible Navier-Stokes equations in cylindrical coordinates read

$$
\begin{aligned}
\rho\left(\frac{\partial \boldsymbol u}{\partial t} + (\boldsymbol u \cdot \nabla)\boldsymbol u\right) &= -\nabla p + \mu \Delta \boldsymbol u, \\
\nabla \cdot \boldsymbol u &= 0,
\end{aligned}
$$

with density $\rho$ and dynamic viscosity $\mu$. The velocity field is
$\boldsymbol u = (u_r, u_\theta, u_z)$ and $p$ is the pressure.

The non-dimensional control parameters are

- The **Reynolds number** of the inner cylinder
  $\mathrm{Re}_1 = \rho\, \omega_1\, r_\mathrm{in}\, (r_\mathrm{out} - r_\mathrm{in}) / \mu$.
- The **radius ratio** $\eta = r_\mathrm{in}/r_\mathrm{out}$.
- The **aspect ratio** $\Gamma = L / (r_\mathrm{out} - r_\mathrm{in})$, where
  $L$ is the axial length of the domain.
- The **outer-cylinder Reynolds number**
  $\mathrm{Re}_2 = \rho\, \omega_2\, r_\mathrm{out}\, (r_\mathrm{out} - r_\mathrm{in}) / \mu$.

## Domain and boundary conditions

The computational domain is the open annulus
$\Omega = \{(r,\theta,z) \mid r_\mathrm{in} < r < r_\mathrm{out},\; 0 < z < L\}$
in a 2D $(r,z)$ cross-section. The boundary conditions are

$$
\begin{aligned}
\boldsymbol u(r=r_\mathrm{in}) &= (0,\; \omega_1 r_\mathrm{in},\; 0), \\
\boldsymbol u(r=r_\mathrm{out}) &= (0,\; \omega_2 r_\mathrm{out},\; 0), \\
\boldsymbol u(z=0) &= \boldsymbol u(z=L) \quad \text{(periodic)}, \\
p &\text{ pinned at one point to remove the null space.}
\end{aligned}
$$

## Analytical reference: circular Couette flow

For steady, axisymmetric, purely azimuthal flow the analytical solution is

$$
u_\theta^\mathrm{Couette}(r) = A r + \frac{B}{r},
$$

with constants

$$
\begin{aligned}
A &= \frac{\omega_2 r_\mathrm{out}^2 - \omega_1 r_\mathrm{in}^2}{r_\mathrm{out}^2 - r_\mathrm{in}^2}, \\
B &= \frac{(\omega_1 - \omega_2) r_\mathrm{in}^2 r_\mathrm{out}^2}{r_\mathrm{out}^2 - r_\mathrm{in}^2}.
\end{aligned}
$$

This is the reference used to compute the relative $L^2$ error of the velocity
field in the postprocessing step.

## Output metrics

- `l2_error_velocity_rel` — relative $L^2$ error of the velocity field with
  respect to the analytical Couette solution.
- `solution_metrics.json` — per-configuration metrics, written by the
  postprocessing script.

## Table of parameters

### Model parameters

| Parameter             | Description                                |
| --------------------- | ------------------------------------------ |
| $r_\mathrm{in}$[m]    | Inner cylinder radius.                     |
| $r_\mathrm{out}$[m]   | Outer cylinder radius.                     |
| $L$[m]                | Axial length of the domain.                |
| $\rho$[kg/m³]         | Fluid density.                             |
| $\mu$[Pa·s]           | Dynamic viscosity.                         |
| $\omega_1$[rad/s]     | Inner-cylinder angular velocity.           |
| $\omega_2$[rad/s]     | Outer-cylinder angular velocity.           |

### Numerical parameters

| Parameter         | Description                                |
| ----------------- | ------------------------------------------ |
| mesh refinement   | Number / size of cells per configuration.  |
| solver tolerances | Linear and non-linear solver tolerances.   |
| time step         | Time step (for transient runs).           |

## Numerical Results

The generated notebook `notebooks/rotating_cylinders.ipynb` is rebuilt by the
`merge-docs-to-notebooks` GitHub Actions workflow and is uploaded as an
artifact on every push to `main`, PR to `main`, and manual dispatch.


# Postprocessing for the rotating cylinders benchmark

This notebook runs the postprocessing logic live. It is regenerated by
the `merge-docs-to-notebooks` GitHub Actions workflow from the benchmark
documentation in `docs/rotating-cylinders.md`.


## 1. Imports

In [ ]:
import json
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "dumux"))


## 2. Locate benchmark result files

Each configuration of the DuMux benchmark produces a folder under
`dumux/rotating-cylinders/results/<configuration>/` containing a
`test_rotatingcylinders.zip` archive with a `solution_metrics.json` file
inside.

In [ ]:
RESULTS_DIR = Path("dumux/rotating-cylinders/results")
METRIC_FILES = []
if RESULTS_DIR.exists():
    for conf_dir in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
        zip_path = conf_dir / "test_rotatingcylinders.zip"
        if zip_path.exists():
            with zipfile.ZipFile(zip_path) as zf:
                for member in zf.namelist():
                    if member.endswith("solution_metrics.json"):
                        with zf.open(member) as f:
                            METRIC_FILES.append((conf_dir.name, json.load(f)))
print(f"Found {len(METRIC_FILES)} metric files.")


## 3. Run the convergence test

This is the same logic as `dumux/convergence_test.py`, inlined so the
notebook is self-contained and runs end-to-end.

In [ ]:
data = []
for conf, metrics in METRIC_FILES:
    data.append({
        "conf": conf,
        "velocity_error": metrics.get("l2_error_velocity_rel"),
    })
df = pd.DataFrame(data)
try:
    df["conf_sort"] = pd.to_numeric(df["conf"])
    df = df.sort_values("conf_sort")
except ValueError:
    df = df.sort_values("conf")
df


## 4. Convergence plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["conf"], df["velocity_error"], marker="s",
        label=r"Relative $L^2$ Error for Velocity")
ax.set_yscale("log")
ax.set_xlabel("Configuration")
ax.set_ylabel("Relative Error")
ax.set_title("Convergence Summary")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.xticks(rotation=45)
fig.tight_layout()
plt.show()


## 5. Analytical Couette reference

The analytical Couette velocity profile (see `docs/rotating-cylinders.md`)
is used as the reference for the relative L² error. We re-derive it here
for verification.

In [ ]:
def couette_profile(r, r_in, r_out, omega_1, omega_2):
    A = (omega_2 * r_out**2 - omega_1 * r_in**2) / (r_out**2 - r_in**2)
    B = (omega_1 - omega_2) * r_in**2 * r_out**2 / (r_out**2 - r_in**2)
    return A * r + B / r

r = np.linspace(0.5, 1.0, 200)
u_theta = couette_profile(r, r_in=0.5, r_out=1.0, omega_1=1.0, omega_2=0.0)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r, u_theta)
ax.set_xlabel("r")
ax.set_ylabel(r"$u_\\theta$")
ax.set_title("Analytical Couette profile (omega_1=1, omega_2=0)")
ax.grid(True, ls="--", alpha=0.3)
fig.tight_layout()
plt.show()
